# file_system

In [ ]:
open sm'_operators
open rust
open rust_operators

In [ ]:
///- --test static

open testing

## fsharp

### file_mode

In [ ]:
nominal file_mode' = $'System.IO.FileMode'

union file_mode =
    | ModeCreateNew
    | ModeCreate
    | ModeOpen
    | ModeOpenOrCreate
    | Truncate
    | Append

inl file_mode = function
    | ModeCreateNew => $'System.IO.FileMode.CreateNew' : file_mode'
    | ModeCreate => $'System.IO.FileMode.Create' : file_mode'
    | ModeOpen => $'System.IO.FileMode.Open' : file_mode'
    | ModeOpenOrCreate => $'System.IO.FileMode.OpenOrCreate' : file_mode'
    | Truncate => $'System.IO.FileMode.Truncate' : file_mode'
    | Append => $'System.IO.FileMode.Append' : file_mode'

### file_access

In [ ]:
nominal file_access' = $'System.IO.FileAccess'

union file_access =
    | AccessRead
    | AccessWrite
    | AccessReadWrite

inl file_access = function
    | AccessRead => $'System.IO.FileAccess.Read' : file_access'
    | AccessWrite => $'System.IO.FileAccess.ReadWrite' : file_access'
    | AccessReadWrite => $'System.IO.FileAccess.ReadWrite' : file_access'

### file_share

In [ ]:
nominal file_share' = $'System.IO.FileShare'

union file_share =
    | ShareNone
    | ShareRead
    | ShareWrite
    | ShareReadWrite
    | ShareDelete

inl file_share = function
    | ShareNone => $'System.IO.FileShare.None' : file_share'
    | ShareRead => $'System.IO.FileShare.Read' : file_share'
    | ShareWrite => $'System.IO.FileShare.Write' : file_share'
    | ShareReadWrite => $'System.IO.FileShare.ReadWrite' : file_share'
    | ShareDelete => $'System.IO.FileShare.Delete' : file_share'

### file_stream

In [ ]:
nominal file_stream' = $'System.IO.FileStream'

inl file_stream (path : string) mode access share : file_stream' =
    run_target function
        | Fsharp => fun () =>
            inl mode = mode |> file_mode
            inl access = access |> file_access
            inl share = share |> file_share
            $'new System.IO.FileStream (!path, !mode, !access, !share)'
        | _ => fun () => null ()

### file_info

In [ ]:
nominal file_info =
    `(
        backend_switch `(()) `({}) {
            Fsharp = (fun () =>
                global "type System_IO_FileInfo = System.IO.FileInfo\n"
                        ) : () -> ()
        }
        $'' : $'System_IO_FileInfo'
    )

inl file_info (path : string) : file_info =
    run_target function
        | Fsharp => fun () => $'!path |> `file_info '
        | _ => fun () => null ()

### directory_info

In [ ]:
nominal directory_info =
    `(
        backend_switch `(()) `({}) {
            Fsharp = (fun () =>
                global "type System_IO_DirectoryInfo = System.IO.DirectoryInfo\n"
                        ) : () -> ()
        }
        $'' : $'System_IO_DirectoryInfo'
    )

inl directory_info (path : string) : directory_info =
    run_target function
        | Fsharp => fun () => $'!path |> `directory_info '
        | _ => fun () => null ()

### directory_info_exists

In [ ]:
inl directory_info_exists (info : directory_info) : bool =
    run_target function
        | Fsharp => fun () => info |> $'_.Exists'
        | _ => fun () => null ()

### directory_info_creation_time

In [ ]:
inl directory_info_creation_time (info : directory_info) : date_time.date_time =
    run_target function
        | Fsharp => fun () => info |> $'_.CreationTime'
        | _ => fun () => null ()

### directory_info_name

In [ ]:
inl directory_info_name (info : directory_info) : string =
    run_target function
        | Fsharp => fun () => info |> $'_.Name'
        | _ => fun () => null ()

### directory_info_full_name

In [ ]:
inl directory_info_full_name (info : directory_info) : string =
    run_target function
        | Fsharp => fun () => info |> $'_.FullName'
        | _ => fun () => null ()

### file_attributes

In [ ]:
nominal file_attributes = $'System.IO.FileAttributes'

### directory_info_attributes

In [ ]:
let directory_info_attributes (info : directory_info) : file_attributes =
    run_target function
        | Fsharp => fun () => info |> $'_.Attributes'
        | _ => fun () => null ()

### file_attributes_reparse_point

In [ ]:
let file_attributes_reparse_point () : file_attributes =
    run_target function
        | Fsharp => fun () => $'`file_attributes.ReparsePoint'
        | _ => fun () => null ()

### file_attributes_has_flag

In [ ]:
let file_attributes_has_flag (flag : file_attributes) (file_attributes : file_attributes) : bool =
    run_target function
        | Fsharp => fun () => $'!file_attributes.HasFlag !flag '
        | _ => fun () => null ()

### create_directory

In [ ]:
let create_directory (path : string) : directory_info =
    run_target function
        | Fsharp => fun () => path |> $'System.IO.Directory.CreateDirectory'
        | _ => fun () => null ()

### directory_get_files

In [ ]:
let directory_get_files (path : string) : array_base string =
    run_target function
        | Fsharp => fun () => path |> $'System.IO.Directory.GetFiles'
        | _ => fun () => null ()

### file_move

In [ ]:
let file_move (new_path : string) (old_path : string) : () =
    run_target function
        | Fsharp => fun () => $'System.IO.File.Move (!old_path, !new_path)'
        | _ => fun () => ()

### write_all_text_async

In [ ]:
let write_all_text_async (path : string) (text : string) : _ () =
    run_target function
        | Fsharp => fun () => $'System.IO.File.WriteAllTextAsync (!path, !text)' |> async.await_task
        | _ => fun () => null ()

### file_system_info

In [ ]:
nominal file_system_info = $'System.IO.FileSystemInfo'

### get_source_directory

In [ ]:
inl get_source_directory () =
    run_target function
        | Rust => fun () => $'Rc::<str>::from(env\!("CARGO_MANIFEST_DIR"))' : string
        | _ => fun () => $'__SOURCE_DIRECTORY__' : string

In [1]:
///- --test
///> fsharp

get_source_directory ()
|> directory_info
|> directory_info_name
|> _assert_in [ "spiral"; "fsharp" ]

11:29:23 1d _assert_in / { x = spiral; x' = spiral }

## rust

### display

In [ ]:
nominal display =
    `(
        rust_type
            `($'std_path_Display')
            `($'std::path::Display')
            "type std_path_Display = string"
    )

### path

In [ ]:
nominal path =
    `(
        rust_type
            `($'std_path_Path')
            `($'std::path::Path')
            "type std_path_Path = class end"
    )

### path_buf

In [ ]:
nominal path_buf =
    `(
        rust_type
            `($'std_path_PathBuf')
            `($'std::path::PathBuf')
            "type std_path_PathBuf = string"
    )

### new_path_buf

In [ ]:
inl new_path_buf (path : sm'.std_string) : path_buf =
    run_target function
        | Rust => fun () => !\\(path, $'"std::path::PathBuf::from($0)"')
        | _ => fun () => path |> convert.unbox

### path_buf_from

In [ ]:
inl path_buf_from (path : rust.box path) : path_buf =
    !\\(path, $'"std::path::PathBuf::from($0)"')

### path_buf_join

In [ ]:
inl path_buf_join (s : string) (path_buf : path_buf) : path_buf =
    !\\((path_buf, s |> sm'.to_std_string), $'"$0.join($1)"')

### path_buf_strip_prefix

In [ ]:
inl path_buf_strip_prefix (s : string) (path_buf : path_buf) : path_buf =
    !\\((path_buf, s |> sm'.to_std_string), $'"$0.strip_prefix($1).unwrap().to_path_buf()"')

### path_display

In [ ]:
inl path_display (path : rust.ref path) : display =
    !\\(path, $'"$0.display()"')

### path_buf_display

In [ ]:
inl path_buf_display (path_buf : path_buf) : display =
    run_target_args (fun () => path_buf) function
        | Rust => fun path_buf =>
            inl path_buf = join path_buf
            !\($'"!path_buf.display()"')
        | _ => fun path_buf => path_buf |> convert.unbox

### path_buf_file_name

In [ ]:
inl path_buf_file_name (path : path_buf) : optionm'.option' (rust.ref sm'.os_str) =
    !\\(path, $'"$0.file_name()"')

### path_buf_exists

In [ ]:
inl path_buf_exists (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.exists()"')

### path_buf_is_dir

In [ ]:
inl path_buf_is_dir (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.is_dir()"')

### path_buf_is_file

In [ ]:
inl path_buf_is_file (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.is_file()"')

### path_buf_is_symlink

In [ ]:
inl path_buf_is_symlink (path_buf : path_buf) : bool =
    !\\(path_buf, $'"$0.is_symlink()"')

### path_buf_parent

In [ ]:
inl path_buf_parent (path_buf : path_buf) : optionm'.option' path_buf =
    !\\(path_buf, $'"$0.parent().map(std::path::PathBuf::from)"')

### dir_entry

In [ ]:
nominal dir_entry =
    `(
        rust_type
            `($'async_walkdir_DirEntry')
            `($'async_walkdir::DirEntry')
            "type async_walkdir_DirEntry = class end"
    )

### walk_dir

In [ ]:
nominal walk_dir =
    `(
        rust_type
            `($'async_walkdir_WalkDir')
            `($'async_walkdir::WalkDir')
            "type async_walkdir_WalkDir = class end"
    )

### async_walkdir_filtering

In [ ]:
nominal async_walkdir_filtering =
    `(
        rust_type
            `($'async_walkdir_Filtering')
            `($'async_walkdir::Filtering')
            "type async_walkdir_Filtering = class end"
    )

### filtering

In [ ]:
union filtering =
    | Ignore
    | IgnoreDir
    | Continue

### async_walkdir_error

In [ ]:
nominal async_walkdir_error =
    `(
        rust_type
            `($'async_walkdir_Error')
            `($'async_walkdir::Error')
            "type async_walkdir_Error = class end"
    )

### new_walk_dir

In [ ]:
inl new_walk_dir (dir : string) : walk_dir =
    !\\(dir, $'"async_walkdir::WalkDir::new(&*$0)"')

### walk_dir_filter

In [ ]:
inl walk_dir_filter (fn : dir_entry -> async.future_pin_send filtering) (walk_dir : walk_dir) : walk_dir =
    if is_rust () then
        global "struct SpiralAssertSend<T>(T);\nunsafe impl<T> Send for SpiralAssertSend<T> {}\nimpl<T: std::future::Future + Unpin> std::future::Future for SpiralAssertSend<T> {\n    type Output = T::Output;\n    fn poll(mut self: std::pin::Pin<&mut Self>, cx: &mut std::task::Context<'_>) -> std::task::Poll<T::Output> { std::pin::Pin::new(&mut self.0).poll(cx) }\n}"
        inl fn entry =
            inl future = fn entry
            async.new_future_move_send fun () =>
                inl result = future |> async.await_send
                inl filtering : async_walkdir_filtering =
                    match result with
                    | Ignore => !\($'"async_walkdir::Filtering::Ignore"')
                    | IgnoreDir => !\($'"async_walkdir::Filtering::IgnoreDir"')
                    | Continue => !\($'"async_walkdir::Filtering::Continue"')
                filtering
        !\\((walk_dir, fn), $'"{ let f = SpiralAssertSend($1); async_walkdir::WalkDir::filter($0, move |x| { let f = &f; SpiralAssertSend((f.0)(x)) }) }"')
    else
        inl fn entry = async.new_future_send fun () =>
            inl result = fn entry |> async.await_send
            inl filtering : async_walkdir_filtering =
                match result with
                | Ignore => !\($'"async_walkdir::Filtering::Ignore"')
                | IgnoreDir => !\($'"async_walkdir::Filtering::IgnoreDir"')
                | Continue => !\($'"async_walkdir::Filtering::Continue"')
            filtering
        !\\((walk_dir, fn), $'"async_walkdir::WalkDir::filter($0, move |x| $1(x))"')

### file_type

In [ ]:
nominal file_type =
    `(
        rust_type
            `($'std_fs_FileType')
            `($'std::fs::FileType')
            "type std_fs_FileType = class end"
    )

### dir_entry_file_type

In [ ]:
inl dir_entry_file_type (dir_entry : dir_entry) : async.future_pin_send (resultm.result' file_type stream.io_error) =
    inl dir_entry = dir_entry |> rust.emit
    !\($'"Box::pin(async_walkdir::DirEntry::file_type(&!dir_entry))"')

### file_type_is_dir

In [ ]:
inl file_type_is_dir (file_type : file_type) : bool =
    !\\(file_type, $'"std::fs::FileType::is_dir(&$0)"')

### file

In [ ]:
nominal file =
    `(
        rust_type
            `($'std_fs_File')
            `($'std::fs::File')
            "type std_fs_File = class end"
    )

### file_open

In [ ]:
inl file_open (path : string) : resultm.result' file stream.io_error =
    !\($'"std::fs::File::open(&*!path)"')

### rename

In [ ]:
inl rename (to : string) (path : string) : resultm.result' () stream.io_error =
    !\($'"std::fs::rename(&*!path, &*!to)"')

### dir_entry_path

In [ ]:
inl dir_entry_path (dir_entry : dir_entry) : path_buf =
    !\\(dir_entry, $'"async_walkdir::DirEntry::path(&$0)"')

### create_dir_all

In [ ]:
inl create_dir_all (path : string) : resultm.result' () stream.io_error =
    !\\(path, $'"std::fs::create_dir_all(&*$0)"')

### file_info_link_target

In [ ]:
inl file_info_link_target (file_info : file_info) : string =
    run_target function
        | Fsharp => fun () =>
            file_info |> $'_.LinkTarget'
        | _ => fun () => null ()

### read

In [ ]:
inl read (path : string) : resultm.result' (am'.vec u8) stream.io_error =
    !\\(path, $'"std::fs::read(&*$0)"')

## typescript

### ts_path_join

In [ ]:
inl ts_path_join (b : string) (a : string) : string =
    global "import * as spiral_node_path from 'node:path';"
    $'spiral_node_path.join(!a, !b)' : string

## file_system

### (< />)

In [ ]:
let (</>) (a : string) (b : string) : string =
    run_target_args (fun () => a, b) function
        | Rust => fun a, b =>
            a
            |> sm'.to_std_string
            |> new_path_buf
            |> path_buf_join b
            |> path_buf_display
            |> sm'.format'
            |> sm'.from_std_string
        | TypeScript => fun a, b =>
            a |> ts_path_join b
        | Fsharp => fun a, b =>
            $'System.IO.Path.Combine (!a, !b)'
        | _ => fun a, b =>
            if a = "" then b
            elif (a |> sm'.ends_with "/") || (a |> sm'.ends_with "\\") then a ++# b
            else a ++# "/" ++# b

#### path_join_without_a_native_arm (test)

In [2]:
///- --test
///> gleam
///> zig
///> lua

"dir" </> "file" |> _assert_eq "dir/file"
"dir/" </> "file" |> _assert_eq "dir/file"
"" </> "file" |> _assert_eq "file"

.gleam (gleam)
__assert_eq { expected = dir/file }
__assert_eq { expected = dir/file }
__assert_eq { expected = file }
.zig (zig)
__assert_eq { expected = dir/file }
__assert_eq { expected = dir/file }
__assert_eq { expected = file }
.lua (lua)
__assert_eq { expected = dir/file }
__assert_eq { expected = dir/file }
__assert_eq { expected = file }

### get_temp_path

In [ ]:
let get_temp_path () : string =
    run_target function
        | Rust => fun () =>
            !\($'"std::env::temp_dir()"')
            |> path_buf_display
            |> sm'.format'
            |> sm'.from_std_string
        | Fsharp => fun () =>
            $'System.IO.Path.GetTempPath' ()
        | target => fun () => join "file_system.get_temp_path / target: " ++#? target |> failwith

### get_file_name

In [ ]:
let get_file_name (path : string) : string =
    run_target_args' path function
        | Fsharp => fun path =>
            path |> $'System.IO.Path.GetFileName'
        | Rust => fun path =>
            path
            |> sm'.to_std_string
            |> new_path_buf
            |> path_buf_file_name
            |> optionm'.map' sm'.from_os_str_ref
            |> optionm'.unbox
            |> optionm'.default_value ""
        | TypeScript => fun path =>
            global "import * as spiral_node_path from 'node:path';"
            $'spiral_node_path.basename(!path)' : string
        | target => fun path => join "file_system.get_file_name / target: " ++#? target ++# " / path: " ++# path |> failwith

### get_file_name_without_extension

In [ ]:
let get_file_name_without_extension (path : string) : string =
    run_target_args' path function
        | Rust => fun path =>
            inl path_buf = path |> sm'.to_std_string |> new_path_buf
            inl file_stem = !\\(path_buf, $'"$0.file_stem()"')
            match file_stem |> optionm'.map' sm'.from_os_str_ref |> optionm'.unbox with
            | Some file_stem => file_stem
            | None => ""
        | _ => fun path =>
            path |> $'System.IO.Path.GetFileNameWithoutExtension'

### get_directory_name

In [ ]:
let get_directory_name (path : string) : string =
    run_target_args' path function
        | Fsharp _ => fun path =>
            path |> $'System.IO.Path.GetDirectoryName'
        | Rust => fun path =>
            path
            |> sm'.to_std_string
            |> new_path_buf
            |> path_buf_file_name
            |> optionm'.map' sm'.from_os_str_ref
            |> optionm'.unbox
            |> optionm'.default_value ""
        | _ => fun _ => null ()

### get_extension

In [ ]:
let get_extension (path : string) : string =
    run_target_args' path function
        | Rust => fun path =>
            inl path_buf = path |> sm'.to_std_string |> new_path_buf
            !\\(path_buf, $'"$0.extension()"')
            |> optionm'.unwrap
            |> sm'.from_os_str_ref
        | _ => fun path =>
            path |> $'System.IO.Path.GetExtension'

### directory_separator_char

In [ ]:
let directory_separator_char () : char =
    run_target function
        | Rust => fun () => $'(std::path::MAIN_SEPARATOR as u8)' : char
        | TypeScript _ => fun () => '/'
        | Zig => fun () => $'std.fs.path.sep' : char
        | _ => fun () => $'System.IO.Path.DirectorySeparatorChar'

### get_current_directory

In [ ]:
let get_current_directory () : string =
    run_target function
        | Rust => fun () =>
            inl current_dir = !\($'"std::env::current_dir()"') : resultm.result' path_buf stream.io_error
            current_dir
            |> resultm.unwrap'
            |> path_buf_display
            |> sm'.format'
            |> sm'.from_std_string
        | Fsharp => fun () =>
            $'System.IO.Directory.GetCurrentDirectory' ()
        | TypeScript _ => fun () =>
            global "import * as spiral_node_fs from 'node:fs';"
            $'spiral_node_fs.realpathSync(".")' : string
        | Zig => fun () =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(std.process.currentPathAlloc(spiral_io_threaded.io(), spiral_gpa) catch \@panic("file_system.get_current_directory failed"))' : string
        | _ => fun () => null ()

In [3]:
///- --test

get_current_directory ()
|> _assert_contains (directory_separator_char ())

__assert_contains { expected = / }

### directory_exists

In [ ]:
let directory_exists (path : string) : bool =
    run_target_args' path function
        | Zig => fun _ =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(if (std.Io.Dir.cwd().statFile(spiral_io_threaded.io(), !path, .{})) |s| s.kind == .directory else |_| false)' : bool
        | Fsharp => fun path =>
            path |> $'System.IO.Directory.Exists'
        | Rust => fun path =>
            inl path = path |> sm'.to_std_string |> new_path_buf
            path_buf_exists path && path_buf_is_dir path
        | TypeScript => fun path =>
            global "import * as spiral_node_fs from 'node:fs';"
            $'(spiral_node_fs.existsSync(!path) && spiral_node_fs.statSync(!path).isDirectory())' : bool
        | _ => fun _ => null ()

### directory_get_parent

In [ ]:
let directory_get_parent (path : string) : optionm'.option' string =
    run_target_args' path function
        | Fsharp => fun path =>
            inl parent : directory_info = path |> $'System.IO.Directory.GetParent'
            if parent =. null ()
            then None
            else parent |> directory_info_full_name |> Some
            |> optionm'.box
        | Rust => fun path =>
            inl path_buf = path |> sm'.to_std_string |> new_path_buf
            inl parent = path_buf |> path_buf_parent
            parent
            |> optionm'.map' (path_buf_display >> sm'.format' >> sm'.from_std_string)
        | TypeScript _ => fun path =>
            global "import * as spiral_node_path from 'node:path';"
            inl parent = $'spiral_node_path.dirname(!path)' : string
            if parent = path
            then None
            else parent |> Some
            |> optionm'.box
        | _ => fun _ => optionm'.none' ()

### create_temp_path'

In [ ]:
let create_temp_path' (guid : guid.guid) =
    run_target_args' guid function
        | _ => fun guid =>
            get_temp_path ()
            </> join "!create_temp_path_"
            </> (env.get_entry_assembly_name ())
            </> (guid |> sm'.obj_to_string)

In [4]:
///- --test
///> fsharp
///> rust -d chrono

guid.hash_guid ""
|> create_temp_path'
|> _assert_contains (directory_separator_char ())

.fsx (fsharp)
__assert_contains { expected = / }
.rs (rust)
__assert_contains { expected = / }

### create_temp_path

In [ ]:
let create_temp_path () =
    run_target function
        | _ => fun () =>
            date_time.now ()
            |> date_time.new_guid_from_date_time
            |> create_temp_path'

### file_copy

In [ ]:
let file_copy (new_path : string) (old_path : string) : () =
    run_target_args' (old_path, new_path) function
        | Fsharp => fun old_path, new_path =>
            $'System.IO.File.Copy (!old_path, !new_path, true)'
        | Rust => fun old_path, new_path =>
            inl result : _ _ stream.io_error = !\\((old_path, new_path), $'"std::fs::copy(&*$0, &*$1)"')
            match result |> resultm.map_error' sm'.format' |> resultm.unbox with
            | Ok (result : u64) =>
                trace Debug
                    fun () => "file_system.file_copy"
                    fun () => { old_path new_path result }
            | Error error =>
                trace Warning
                    fun () => "file_system.file_copy"
                    fun () => { old_path new_path error }
        | _ => fun _ => ()

### file_exists

In [ ]:
let file_exists (path : string) : bool =
    run_target_args' path function
        | Zig => fun _ =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(if (std.Io.Dir.cwd().statFile(spiral_io_threaded.io(), !path, .{})) |s| s.kind == .file else |_| false)' : bool
        | Fsharp => fun path =>
            path |> $'System.IO.File.Exists'
        | Rust => fun path =>
            inl path_buf = path |> sm'.to_std_string |> new_path_buf
            path_buf_exists path_buf && path_buf_is_file path_buf
        | TypeScript => fun path =>
            global "import * as spiral_node_fs from 'node:fs';"
            $'(spiral_node_fs.existsSync(!path) && spiral_node_fs.statSync(!path).isFile())' : bool
        | Gleam _ => fun path =>
            global "@external(erlang, \"filelib\", \"is_regular\")\nfn spiral_filelib_is_regular(path: String) -> Bool"
            $'spiral_filelib_is_regular(!path)' : bool
        | _ => fun _ => null ()

#### file_exists_on_gleam_tells_files_from_missing_paths (test)

In [5]:
///- --test
///> gleam

file_exists "gleam.toml" |> _assert_eq true
file_exists "spiral-no-such-file.txt" |> _assert_eq false

__assert_eq { expected = true }
__assert_eq { expected = false }

### directory_delete

In [ ]:
let directory_delete recursive (path : string) : () =
    run_target_args' (path, recursive) function
        | Fsharp => fun path, recursive =>
            $'System.IO.Directory.Delete (!path, !recursive)'
        | Rust => fun path, recursive =>
            if path |> directory_exists then
                if recursive
                then !\\(path, $'"std::fs::remove_dir_all(&*$0).unwrap()"')
                else !\\(path, $'"std::fs::remove_dir(&*$0).unwrap()"')
        | _ => fun _ => ()

### write_all_text

In [ ]:
inl write_all_text (path : string) (text : string) : () =
    run_target_args' (path, text) function
        | Zig => fun _ =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'std.Io.Dir.cwd().writeFile(spiral_io_threaded.io(), .{ .sub_path = !path, .data = !text }) catch \@panic("file_system.write_all_text failed")' : ()
        | Fsharp => fun path, text =>
            $'System.IO.File.WriteAllText (!path, !text)'
        | Rust => fun path, text =>
            !\\((path, text), $'"std::fs::write(&*$0, &*$1).unwrap()"')
        | _ => fun _ => ()

### read_all_bytes

In [ ]:
inl read_all_bytes (path : string) : am'.vec u8 =
    run_target function
        | Fsharp => fun () =>
            $'!path |> System.IO.File.ReadAllBytes'
            |> am'.to_vec
        | Rust => fun () =>
            path |> read |> resultm.unwrap'
        | _ => fun () => null ()

### read_all_text

In [ ]:
inl read_all_text (path : string) : string =
    run_target function
        | Zig => fun () =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'(std.Io.Dir.cwd().readFileAlloc(spiral_io_threaded.io(), !path, spiral_gpa, .unlimited) catch \@panic("file_system.read_all_text failed"))' : string
        | Fsharp => fun () =>
            $'!path |> System.IO.File.ReadAllText'
        | Rust => fun () =>
            path
            |> read_all_bytes
            |> sm'.string_from_utf8
            |> resultm.unwrap'
            |> sm'.from_std_string
        | _ => fun () => null ()

### directory_create_symbolic_link

In [ ]:
inl directory_create_symbolic_link (target : string) (path : string) : () =
    run_target function
        | Fsharp => fun () =>
            ($'System.IO.Directory.CreateSymbolicLink (!path, !target)' : file_system_info)
            |> ignore
        | Rust => fun () =>
            (!\\((target, path), $'"true; \#[cfg(windows)] std::os::windows::fs::symlink_dir(&*$0, &*$1).unwrap()"') : bool) |> ignore
            (!\\((target, path), $'"true; \#[cfg(unix)] std::os::unix::fs::symlink(&*$0, &*$1).unwrap()"') : bool) |> ignore
        | _ => fun () => ()

### file_create_symbolic_link

In [ ]:
inl file_create_symbolic_link (target : string) (path : string) : () =
    run_target function
        | Fsharp => fun () =>
            ($'System.IO.File.CreateSymbolicLink (!path, !target)' : file_system_info)
            |> ignore
        | Rust => fun () =>
            (!\\((target, path), $'"true; \#[cfg(windows)] std::os::windows::fs::symlink_file(&*$0, &*$1).unwrap()"') : bool) |> ignore
            (!\\((target, path), $'"true; \#[cfg(unix)] std::os::unix::fs::symlink(&*$0, &*$1).unwrap()"') : bool) |> ignore
        | _ => fun () => ()

### file_type

In [ ]:
union file_type =
    | File
    | Directory

### find_parent

In [ ]:
let find_parent file_type name root_dir =
    inl is_file = file_type = File
    inl exists_fn = if is_file then file_exists else directory_exists
    let rec 루프 dir =
        if dir </> name |> exists_fn
        then dir |> Ok
        else
            inl parent = dir |> directory_get_parent
            match parent |> optionm'.unbox with
            | Some parent => parent |> 루프
            | None =>
                sm'.(++#) "file_system.find_parent / No parent for " (if is_file then "file" else "dir")
                ++# $'$" \'{!name}\' at \'{!root_dir}\' (until \'{!dir}\')"'
                |> fun x => x : string
                |> Error
    root_dir |> 루프

In [6]:
///- --test
///> fsharp

a ;[ Directory, "workspace"; File, "README.md" ]
|> am.map fun file_type, file =>
    get_source_directory ()
    |> find_parent file_type file
    |> resultm.get
    |> directory_info
    |> directory_info_name
|> am'.distinct
|> fun (a x : _ int _) => x
|> _assert_in [ ;[ "spiral" ]; ;[ "polyglot" ] ]

11:29:29 1d _assert_in / { x = [spiral]; x' = [spiral] }

In [7]:
///- --test
///> rust

a ;[ Directory, "workspace"; File, "README.md" ]
|> am.map fun file_type, file =>
    fun () =>
        join
            get_current_directory ()
            |> find_parent file_type file
            |> resultm.get
            |> sm'.to_std_string
            |> new_path_buf
            |> path_buf_file_name
            |> optionm'.try'
            |> sm'.from_os_str_ref
            |> Some
            |> optionm'.box
    |> fun x => x () |> optionm'.unbox
    |> optionm'.default_value ""
|> am'.distinct
|> fun result =>
    result |> am'.length |> _assert_eq 1i32
    index result 0i32 |> _assert_eq "spiral"

__assert_eq { expected = 1 }
__assert_eq { expected = spiral }

0

### file_delete

In [ ]:
inl file_delete (path : string) : () =
    run_target function
        | Zig => fun () =>
            global "var spiral_io_threaded: std.Io.Threaded = .init_single_threaded;"
            $'std.Io.Dir.cwd().deleteFile(spiral_io_threaded.io(), !path) catch {}' : ()
        | Fsharp => fun () =>
            path |> $'System.IO.File.Delete'
        | Rust => fun () =>
            inl result : resultm.result' () stream.io_error =
                if path |> file_exists
                then !\\(path, $'"std::fs::remove_file(&*$0)"')
                else Ok () |> resultm.box
            match result |> resultm.map_error' sm'.format |> resultm.unbox with
            | Ok () => ()
            | Error error' => trace Critical (fun () => "file_system.file_delete") fun () => { error' }
        | _ => fun () => ()

### read_link

In [ ]:
let read_link (path : string) : resultm.result' path_buf stream.io_error =
    let run 루프 n error path' =
        inl name = path' |> get_file_name
        inl parent = path' |> directory_get_parent |> optionm'.unbox
        inl error'' = #?error
        match parent with
        | _ when n >= 11 =>
            ($'$"file_system.read_link / "' : string)
            ++# $'$"path: {!path} / n: {!n} / path\': {!path'} / name: {!name}"'
            |> stream.new_io_error
            |> resultm.err
        | Some parent when path' <>. "" =>
            match 루프 (n + 1) parent |> resultm.map_error' sm'.format |> resultm.unbox with
            | Ok parent' =>
                (parent' |> path_buf_display |> convert.convert) </> name
                |> sm'.to_std_string
                |> new_path_buf
                |> resultm.ok''
            | Error error' =>
                ($'$"file_system.read_link / "' : string)
                ++# $'$"error\': {!error'} / error: {!error''} / name: {!name}"'
                |> stream.new_io_error
                |> resultm.err
        | _ =>
            ($'$"file_system.read_link / run / The file or directory is not a reparse point. / "' : string)
            ++# $'$"path: {!path} / error: {!error''} / path\': {!path'} / name: {!name}"'
            |> stream.new_io_error
            |> resultm.err

    run_target function
        | Rust => fun () =>
            if path |> directory_exists
            then !\\(path, $'"std::fs::read_link(&*$0)"')
            else
                let rec 루프 forall t {number}.
                    (n : t)
                    (path' : string)
                    : resultm.result' path_buf stream.io_error
                    =
                    run_target function
                        | Rust => fun () =>
                            inl result : _ _ stream.io_error = !\\(path', $'"std::fs::read_link(&*$0)"')
                            inl result = result |> resultm.map_error' sm'.format |> resultm.unbox
                            match result with
                            | Ok x => x |> resultm.ok''
                            | Error error => path' |> run 루프 n error
                        | _ => fun () => null ()
                path |> 루프 0u8
        | Fsharp _
        | TypeScript _ => fun () =>
            let rec 루프 forall t {number}.
                (n : t)
                (path' : string)
                : resultm.result' path_buf stream.io_error
                =
                inl result =
                    path'
                    |> directory_info
                    |> directory_info_attributes
                    |> file_attributes_has_flag (file_attributes_reparse_point ())
                if result then
                    path'
                    |> file_info
                    |> file_info_link_target
                    |> convert.unbox
                    |> resultm.ok''
                else
                    inl error =
                        ($'$"file_system.read_link / Fsharp / "' : string)
                        ++# $'$"The file or directory is not a reparse point. / "'
                        ++# $'$"path: {!path} / result: {!result} / path\': {!path'} / n: {!n}"'
                        |> stream.new_io_error
                    path' |> run 루프 n error
            path |> 루프 0u8
        | _ => fun () => $'Unchecked.defaultof<_>'

### normalize_path

In [ ]:
let normalize_path (path : string) : string =
    if path = ""
    then ""
    else
        inl result =
            match path |> read_link |> resultm.ok' |> optionm'.unbox with
            | Some path_buf =>
                path_buf
                |> path_buf_display
                |> convert.convert
            | None => path
        inl result =
            if result = ""
            then path
            else result
        inl result' = result |> sm'.replace_regex @"^\\\\\?\\" ""
        if ##result' < 2
        then path
        else sm'.(++#) (result' |> sm'.slice 0i32 0i32 |> sm'.to_lower) (result' |> sm'.slice 1i32 ((sm'.length result' : i32) - 1i32)) |> sm'.replace "\\" "/"

### get_workspace_root

In [ ]:
inl get_workspace_root () =
    inl repo_root = env.get_environment_variable "SPIRAL_REPO_ROOT"
    if repo_root <> ""
    then repo_root |> normalize_path
    else
        inl dirs =
            [ get_current_directory; get_source_directory; fun () => "/workspaces" ]
        let run dir =
            dir
            |> find_parent Directory ("spiral" </> "workspace")
            |> function
                | Ok dir => dir |> normalize_path |> Some
                | Error error =>
                    trace Warning
                        fun () => "file_system.get_workspace_root"
                        fun () => { dir error }
                    None
        inl root =
            (None, dirs)
            ||> listm.fold fun acc dir =>
                match acc with
                | Some dir => Some dir
                | None => dir () |> run
            |> optionm.value
        inl root =
            match root |> get_file_name with
            | "deps" =>
                root
                |> directory_get_parent
                |> optionm'.get'
                |> run
                |> optionm.value
            | _ => root
        root </> "spiral"

### get_workspace_root_external

In [ ]:
inl get_workspace_root_external () =
    inl workspace_root = get_workspace_root ()
    inl current_dir = get_current_directory () |> sm'.to_lower
    inl workspace_root = workspace_root |> sm'.to_lower
    if current_dir |> sm'.starts_with workspace_root
    then Error workspace_root
    else Ok workspace_root

In [8]:
///- --test
///> rust -d regex

match get_workspace_root_external () with
| Ok root => root
| Error root => root
|> get_file_name
|> _assert_eq "spiral"

__assert_eq { expected = spiral }

### get_full_path

In [ ]:
let get_full_path (path : string) : string =
    run_target_args (fun () => path) function
        | Fsharp => fun path =>
            path |> $'System.IO.Path.GetFullPath'
        | Rust => fun path =>
            inl path_buf = path |> sm'.to_std_string |> new_path_buf
            if path_buf |> path_buf_exists |> not then
                inl current_dir = get_current_directory ()
                current_dir </> path
                |> normalize_path
                |> sm'.split "/"
                |> fun x =>
                    ((a x : _ i32 _), (0i32, (a ;[] : _ i32 _)))
                    ||> am.foldBack fun x level, acc =>
                        match x, level with
                        | "..", _ => level + 1, acc
                        | ".", _ => level, acc
                        | _, 0 when x |> sm'.ends_with ":" => 0, a ;[ sm'.(++#) (current_dir |> sm'.slice 0i32 0i32) ":" ] ++ acc
                        | _, 0 => 0, a ;[ x ] ++ acc
                        | _ => level - 1, acc
                |> snd
                |> seq.of_array'
                |> sm'.concat (directory_separator_char () |> sm'.obj_to_string)
            else
                inl path : resultm.result' path_buf stream.io_error =
                    !\\(path, $'"std::fs::canonicalize(&*$0)"')
                path
                |> resultm.unwrap'
                |> path_buf_display
                |> sm'.format'
                |> sm'.from_std_string
        | _ => fun _ => null ()

In [9]:
///- --test
///> fsharp

"."
|> get_full_path
|> directory_info
|> directory_info_name
|> _assert_in [ "spiral"; "fsharp" ]

11:29:33 1d _assert_in / { x = spiral; x' = spiral }

In [10]:
///- --test
///> rust -d regex

"dir/.././._file"
|> get_full_path
|> _assert_eq (get_current_directory () </> "._file")

__assert_eq { expected = /home/runner/work/spiral/spiral/apps/kino/._file }

In [11]:
///- --test
///> rust -d regex

inl directory_name path =
    path
    |> sm'.to_std_string
    |> new_path_buf
    |> path_buf_file_name
    |> optionm'.unwrap
    |> sm'.from_os_str_ref
"."
|> get_full_path
|> directory_name
|> _assert_eq (get_current_directory () |> directory_name)

__assert_eq { expected = kino }

In [12]:
///- --test
///> rust -d regex

"dir/.././._file"
|> get_full_path
|> normalize_path
|> _assert_eq (get_current_directory () </> "._file" |> normalize_path)

__assert_eq { expected = /home/runner/work/spiral/spiral/apps/kino/._file }

### standardize_path

In [ ]:
let standardize_path path =
    path |> get_full_path |> normalize_path

### absolute_path

In [ ]:
let absolute_path path =
    inl current_dir = get_current_directory ()
    current_dir </> path |> standardize_path

### new_file_uri

In [ ]:
inl new_file_uri (path : string) : string =
    inl path = path |> sm'.trim_start [ '/' ]
    $'$"file:///{!path}"'

In [13]:
///- --test
///> rust -d regex

@"\\?\C:\test"
|> normalize_path
|> new_file_uri
|> _assert_eq "file:///c:/test"

__assert_eq { expected = file:///c:/test }

In [14]:
///- --test
///> rust -d regex

@"\\?\C:\test"
|> normalize_path
|> new_file_uri
|> _assert_eq "file:///c:/test"

__assert_eq { expected = file:///c:/test }

## fsharp

### trace_file

In [ ]:
let rec trace_file text : () =
    run_target function
    | Fsharp => fun () =>
        resultm.from_try
            fun () =>
                inl assembly_name = env.get_entry_assembly_name ()
                inl guid = date_time.now () |> date_time.new_guid_from_date_time
                inl file_name = $'$"{!assembly_name}_{!guid}.txt"'

                inl workspace_root = get_workspace_root ()
                inl trace_dir = workspace_root </> "target/trace"
                trace_dir |> create_directory |> ignore
                inl path = trace_dir </> file_name
                text |> write_all_text_async path |> async.run_synchronously
            fun ex =>
                inl ex = ex ()
                inl text = $'$"file_system.trace_file / ex: %A{!ex}"'
                text |> console.write_line
                text |> trace_file
                ex
        |> resultm.get
    | _ => fun () => ()

In [15]:
///- --test
///> fsharp

inl get_count dir : i64 =
    inl files = dir |> directory_get_files
    a files |> am'.length

inl trace_dir = get_workspace_root () </> "target/trace"
trace_dir |> create_directory |> ignore

inl count = get_count trace_dir

trace_file "test"

get_count trace_dir
|> _assert_eq (count + 1)

__assert_eq { expected = 1 }

### init_trace_file

In [ ]:
inl init_trace_file enabled =
    inl state_trace_file = get_trace_state_or_init None .trace_file
    state_trace_file <- if enabled then trace_file else ignore

## file_system

### create_dir

In [ ]:
let create_dir dir =
    run_target_args' dir function
        | Rust => fun dir =>
            match dir |> create_dir_all |> resultm.map_error' sm'.format' |> resultm.unbox with
            | Ok () =>
                trace Verbose
                    fun () => "file_system.create_dir"
                    fun () => { dir }
            | Error error =>
                trace Critical
                    fun () => "file_system.create_dir"
                    fun () => { dir error }
            inl disposable : _ () = new_disposable fun () =>
                dir
                |> directory_delete true
            disposable
        | _ => fun _ => null ()

### create_temp_dir

In [ ]:
inl create_temp_dir () =
    inl dir = create_temp_path ()
    dir, dir |> create_dir

In [16]:
///- --test
///> rust -d chrono

inl path, disposable = create_temp_dir ()
join
    path
    |> directory_exists
    |> _assert_eq true
    disposable |> use |> ignore
    path
    |> directory_exists
    |> _assert_eq true
path
|> directory_exists
|> _assert_eq false

11:29:38 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_8dda7c442656f7efb85f528b62483421ee2bd98c15d25f91a49346f6b0bc47f1/20261017-1129-3881-5080-000000f40616 }
__assert_eq { expected = true }
__assert_eq { expected = true }
__assert_eq { expected = false }

### create_temp_dir'

In [ ]:
inl create_temp_dir' (hash : string) =
    inl dir = hash |> guid.hash_guid |> create_temp_path'
    dir, dir |> create_dir

### link_directory

In [ ]:
let link_directory target_path path =
    if target_path |> directory_exists |> not
    then target_path |> create_dir |> ignore

    inl lib_dir_path = path |> directory_get_parent |> optionm'.default_value' ""
    if lib_dir_path |> directory_exists |> not
    then lib_dir_path |> create_dir |> ignore

    if (path |> directory_exists)
        && (path |> read_link |> resultm.is_err)
    then path |> directory_delete true

    if path |> directory_exists |> not
    then path |> directory_create_symbolic_link target_path

### link_file

In [ ]:
let link_file target_path path =
    if (path |> file_exists)
        && (path |> read_link |> resultm.is_err)
    then path |> file_delete

    if path |> file_exists |> not
    then path |> file_create_symbolic_link target_path

In [17]:
///- --test
///> rust -d sha2 regex

inl file_name = "LICENSE"
inl text = file_name

inl test_hash =
    #?*(file_name, text)
    |> crypto.hash_text

inl workspace_root = get_workspace_root ()
inl test_dir = workspace_root </> "target/test/file_system" </> test_hash

inl disposable = test_dir |> create_dir
disposable |> use |> ignore

inl dir_path = test_dir </> "dir1"

if dir_path |> directory_exists
then dir_path |> directory_delete true

dir_path |> create_dir |> ignore

inl path = dir_path </> file_name
text |> write_all_text path

inl dir_link_path = test_dir </> "link1"

dir_link_path |> link_directory dir_path

inl link_path = dir_link_path </> file_name

link_path
|> read_all_text
|> _assert_eq text

dir_link_path
|> read_link
|> resultm.unwrap'
|> path_buf_display
|> convert.convert
|> _assert sm'.ends_with "dir1"

link_path
|> read_link
|> resultm.unwrap'
|> path_buf_display
|> convert.convert
|> _assert sm'.ends_with "LICENSE"

inl link_name = "LICENSE_"

inl link_path = dir_path </> link_name

link_path |> link_file path

inl link_path' = dir_link_path </> link_name

link_path'
|> read_all_text
|> _assert_eq text

link_path
|> read_link
|> resultm.unwrap'
|> path_buf_display
|> convert.convert
|> _assert sm'.ends_with "LICENSE"

link_path'
|> read_link
|> resultm.unwrap'
|> path_buf_display
|> convert.convert
|> _assert sm'.ends_with "LICENSE"

11:29:41 1v file_system.create_dir / { dir = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38 }
11:29:41 2v file_system.create_dir / { dir = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1 }
__assert_eq { expected = LICENSE }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1 }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1/LICENSE }
__assert_eq { expected = LICENSE }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675c516636a3bb2c28cd38/dir1/LICENSE }
__assert { expected = /home/runner/work/spiral/spiral/target/test/file_system/d2a575f48045e269d7c0d7a17d801aad61db34f865675

## rust

### file_exists_content

In [ ]:
let file_exists_content path content : bool =
    run_target function
        | Rust => fun () =>
            if path |> file_exists |> not
            then false
            else
                inl existing_content = path |> read_all_text
                content = existing_content
        | _ => fun () => null ()

### write_all_text_exists

In [ ]:
let write_all_text_exists path contents =
    inl exists' = contents |> file_exists_content path
    if not exists' then
        inl dir = path |> directory_get_parent |> optionm'.default_value' ""
        if dir |> directory_exists |> not
        then dir |> create_dir |> ignore
        contents |> write_all_text path

## main

In [18]:
///> fsharp

inl main () =
    init_trace_state None
    $'let file_exists_content x = !file_exists_content x' : ()
    $'let write_all_text_async x = !write_all_text_async x' : ()
    $'let create_temp_path () = !create_temp_path ()' : ()
    $'let create_temp_dir () = !create_temp_dir ()' : ()
    $'let create_temp_dir\' x = !create_temp_dir' x' : ()
    $'let get_source_directory () = !get_source_directory ()' : ()
    $'let normalize_path x = !normalize_path x' : ()
    $'let get_full_path x = !get_full_path x' : ()
    $'let standardize_path x = !standardize_path x' : ()
    $'let new_file_uri x = !new_file_uri x' : ()
    $'let get_workspace_root () = !get_workspace_root ()' : ()
    $'let link_directory x = !link_directory x' : ()
    inl combine x = (</>) x
    $'let (</>) x = !combine x' : ()